# Collaborative Filtering Recommender System - Project Part II

# 1. Introduction and Setup

## 1.1 Overview 🧭

In this project you build a **collaborative filtering (CF) recommender** for a simulated online shop with 40 customers and 50 products 🎉

Every day the shop shows each customer K = 5 products. Your recommender should pick the products that earn the most: products the customer is likely to buy **and** that have a high margin.

The model learns low-dimensional embeddings for customers and products from purchase feedback (logistic matrix factorization) and uses them to predict the purchase probability κ(c,i) for every customer–product pair.

**The challenge: cold start.** During training each customer only sees 15 of the 50 products, but the model has to predict κ for all products, including the ones a customer has never seen.

**How this notebook is organised**
1. Introduction and Setup: setup and a first look at the data
2. Implementing the CF Recommender: the model and your code
3. Training and Validation: choose hyperparameters, train, read the learning curves
4. Test: 100 days against two baselines (Random and Highest Margin)
5. Conclusion and Next Steps

**Your tasks**

| TODO | Section | What you do |
|---|---|---|
| TODO 1 | 2.2 | `_kappa_hat`: predict the purchase probability |
| TODO 2 | 2.3 | `recommend`: pick the top-K products by expected profit |
| TODO 3 | 2.4 | `update_from_session`: set the targets for purchases and non-purchases |
| TODO 4 | 2.5 | `_update_embeddings`: one gradient descent step |
| TODO 5 | 2.6 | Run the check and fix TODO 1–4 until all tests pass |
| TODO 6 | 3.1 | Choose the hyperparameters; revisit them in 5.2 |


## 1.2 Setup

Run the cells in this section once, then collapse it. You don't need to change anything here. In Google Colab, the first cell downloads the project and installs the packages.

The setup loads the shop data (customers, and products with price and cost) and computes the **true purchase probability** κ(c,i) for all 40 × 50 customer–product pairs. Your model never sees these values: they are only used to simulate purchases and to measure how good your predictions are.


In [ ]:
#@title Setup: download project data and install packages
import os
REPO_URL = "https://github.com/eth-ainit-hs26/w2-project-public.git" 
!git clone -q {REPO_URL} project2
os.chdir("project2")
print("Ready. Working folder:", os.getcwd())

In [ ]:
#@title Setup: environment, imports and data — run, don't modify {display-mode: "form"}
from utils.setup import setup_environment
setup_environment()

# Import scientific libraries
import numpy as np

# ▶️ Run this to import all required modules
# Import project modules and utilities
from agents.recommender_agent import RecommenderAgent
from agents.random_recommender import RandomRecommenderAgent
from agents.highest_margin_recommender import HighestMarginRecommenderAgent
from typing import List

# Import all CF analysis utilities
from utils.setup import (
    initialize_system_components,
    prepare_ground_truth_data,
    check_todo_implementations,
)
from utils.training import (
    item_margins,
    training_mask,
    oracle_expected_revenue,
    select_training_items,
    validation_reference_revenues,
    evaluate_validation,
)
from utils.simulation import (
    detailed_agent_simulation,
    print_comprehensive_analysis,
    clone_cf_agent,
    run_exploration,
    print_exploration_runs,
)
from utils.plots import (
    plot_data_overview,
    plot_learning_curves,
    analyze_and_plot_kappa_prediction_quality,
    plot_comprehensive_performance_analysis,
    plot_exploration_runs,
)

In [ ]:
# ▶️ Run this to initialize the database and load customer/item data
# Initialize system components with hyperparameters
customer_registry, item_catalogue, transaction_registry, hyperparams = initialize_system_components(
    n_iterations=300,
    n_simulation_days=100,
    k=5,
    items_per_customer=15
)

# Extract hyperparameters for convenience
N_ITERATIONS = hyperparams['N_ITERATIONS']
N_SIMULATION_DAYS = hyperparams['N_SIMULATION_DAYS']
K = hyperparams['K']
ITEMS_PER_CUSTOMER = hyperparams['ITEMS_PER_CUSTOMER']


In [ ]:
#@title Ground Truth Computation — run, don't modify
# ▶️ Run this to compute the ground truth κ values used for training, validation and testing
customer_ids, item_ids, true_kappa_matrix = prepare_ground_truth_data(customer_registry, item_catalogue)

## 1.3 The Data 🔍

Before building the recommender, take a first look at the shop data. The top row shows the 50 products: the distribution of prices, the distribution of margins (price − cost), and price against margin. The bottom row shows the 40 customers: the distribution of annual income (log scale), income per customer segment (each dot is one customer), and the distribution of age.


In [ ]:
#@title Data Overview — run, don't modify
# ▶️ Run this to plot the product and customer data
plot_data_overview(customer_registry, item_catalogue)

# 2. Implementing the CF Recommender 🛠️

## 2.1 The Model 🧩

The CF recommender describes every customer c by an embedding $U_c \in \mathbb{R}^d$ and every product i by an embedding $V_i \in \mathbb{R}^d$, plus a bias per customer ($b_c$) and per product ($b_i$). From these it predicts the purchase probability $\hat{\kappa}(c,i)$ (TODO 1).

It then shows each customer the products with the highest **expected profit**, margin × $\hat{\kappa}$ (TODO 2).

After each session the model learns from the outcome: purchased products should get a higher $\hat{\kappa}$, products that were shown but not purchased a lower one (TODO 3). Embeddings and biases are adjusted by gradient descent on the logistic loss (TODO 4).

Each of the four cells below defines one method of `RecommenderAgent`. In 2.6 they are attached to the class and tested.


## 2.2 TODO 1: Predict Purchase Probability 🔮

**Function**: `_kappa_hat(self, customer_id: int, item_id: int) -> float`

Predict the probability that customer c buys product i:
$$\hat{\kappa}(c,i) = \sigma(U_c^T V_i + b_c + b_i), \qquad \sigma(x) = \frac{1}{1 + e^{-x}}$$

**Instructions**:
1. Missing embeddings are initialised for you (`_init_customer`, `_init_item`).
2. Compute $z = U_c^T V_i + b_c + b_i$.
3. The code returns $\sigma(z)$ with the helper `self._sigmoid()`.


In [ ]:
def _kappa_hat(self, customer_id: int, item_id: int) -> float:
    """
    Predict the probability that customer will purchase item using learned embeddings.
    
    TODO: Implement this function following the instructions in the markdown cell above.
    
    Returns:
        float: Purchase probability in range [0, 1]
    """
    if customer_id not in self.U:
        self._init_customer(customer_id)
    if item_id not in self.V:
        self._init_item(item_id)

    # 🎯 TODO: compute z = dot product + biases
    z = ...
    return self._sigmoid(z)
 

## 2.3 TODO 2: Recommend Items 🛒

**Function**: `recommend(self, customer_id: int, k: int = 10) -> List[int]`

Recommend the k products with the highest expected profit:
$$\text{Score}(c,i) = (\text{price}_i - \text{cost}_i) \times \hat{\kappa}(c,i)$$

A product is worth recommending if the customer is likely to buy it (high $\hat{\kappa}$) **and** it earns a high margin.

**Instructions**: The loop over `self.item_catalogue.get_available_items()` and the sorting are given. Compute `score` for each item from `item.price`, `item.cost` and `kappa`.


In [ ]:
def recommend(self, customer_id: int, k: int = 10) -> List[int]:
    """
    Recommend top-K items that maximize expected revenue for the customer.
    
    TODO: Implement this function following the instructions in the markdown cell above.
    
    Args:
        customer_id: ID of customer to recommend items for
        k: Number of items to recommend (default: 10)
        
    Returns:
        List[int]: List of top-K item IDs ranked by expected revenue
    """
    inventory = self.item_catalogue.get_available_items()
    if not inventory:
        return []
    
    scored_items = []
    # 🎯 TODO: score and sort items
    for item in inventory:
        kappa = self._kappa_hat(customer_id, item.pid)
        # TODO: Your implementation of score here
        score = ...
        scored_items.append((item.pid, score))

    # Return top K
    scored_items.sort(key=lambda x: x[1], reverse=True)
    return [item_id for item_id, _ in scored_items[:min(k, len(scored_items))]]



## 2.4 TODO 3: Learn from a Session 💬

**Function**: `update_from_session(self, customer_id: int, shown_items: List[int], purchased_items: List[int]) -> None`

After a session the model learns from implicit feedback, from what the customer bought **and** from what they chose not to buy:
- **Purchased** items: target = 1.0, weight = `self.w_purchase`
- Items **shown but not purchased**: target = 0.0, weight = `self.w_neg`

**Instructions**: The loops that call `_update_embeddings` are given. Set `target_purchased` and `target_not_purchased`.


In [ ]:
def update_from_session(self, customer_id: int, shown_items: List[int], 
                        purchased_items: List[int]) -> None:
    """
    Update model based on customer purchases and non-purchases during a session.
    
    TODO: Implement this function following the instructions in the markdown cell above.
    
    Args:
        customer_id: ID of customer who had the session
        shown_items: List of item IDs shown to customer
        purchased_items: List of item IDs customer actually purchased
    """
    # 🎯 TODO: call _update_embeddings for purchases and non-purchases
    target_purchased = ...
    target_not_purchased = ...

    for item_id in purchased_items:
        self._update_embeddings(customer_id, item_id, target=target_purchased
                                , weight=self.w_purchase)

    # Negative feedback: shown but not purchased items
    for item_id in shown_items:
        if item_id not in purchased_items:
            self._update_embeddings(customer_id, item_id, target=target_not_purchased
                                    , weight=self.w_neg)
 

## 2.5 TODO 4: Gradient Descent Update 📉

**Function**: `_update_embeddings(self, customer_id: int, item_id: int, target: float, weight: float)`

One gradient descent step on the logistic loss for a single customer–product pair. The prediction and the error are already in the code:
$$p = \hat{\kappa}(c,i), \qquad \text{error} = \text{target} - p$$

Update the embeddings and biases:
$$U_c := U_c + \alpha w \cdot \mathrm{error} \cdot V_i - \alpha \lambda U_c$$
$$V_i := V_i + \alpha w \cdot \mathrm{error} \cdot U_c - \alpha \lambda V_i$$
$$b_c := b_c + \alpha w \cdot \mathrm{error} - \alpha \lambda b_c$$
$$b_i := b_i + \alpha w \cdot \mathrm{error} - \alpha \lambda b_i$$

where $\alpha$ = learning rate (`self.lr`), $\lambda$ = regularization (`self.reg`) and $w$ = feedback weight (`weight`).

**Instructions**: Write the four update lines for `self.U[customer_id]`, `self.V[item_id]`, `self.b_c[customer_id]` and `self.b_i[item_id]`.


In [ ]:
def _update_embeddings(self, customer_id: int, item_id: int, target: float, weight: float):
    """
    Update customer and item embeddings using gradient descent on logistic loss.
    
    TODO: Implement this function following the instructions in the markdown cell above.
    
    Args:
        customer_id: ID of customer
        item_id: ID of item
        target: Target value (1.0 for purchase, 0.0 for non-purchase)
        weight: Learning weight (w_purchase or w_neg)
    """
    p = self._kappa_hat(customer_id, item_id)
    err = target - p

    # 🎯 TODO: apply gradient descent updates to U, V, b_c, b_i
    # TODO: Your implementation of the gradient updates here
    self.U[customer_id] += ...
    self.V[item_id] += ...
    self.b_c[customer_id] += ...
    self.b_i[item_id] += ...


## 2.6 TODO 5: Check Your Implementation ✅

The cell below attaches your four functions to `RecommenderAgent` and runs five quick tests on a separate test agent: κ̂ lies in [0, 1], `recommend()` returns valid product IDs, both update functions run, and κ̂ changes after an update. This is a functionality check, not a performance test.

**Your task**: If a test fails, fix the corresponding TODO, re-run its cell and then this one, until all tests pass.


In [ ]:
#@title TODO 5: apply methods and check — run, fix TODO 1–4 if needed
# ▶️ Run this to apply your implementations to the RecommenderAgent class
# Apply the implemented methods to RecommenderAgent class
# (This connects your implementations to the actual agent)
RecommenderAgent._kappa_hat = _kappa_hat
RecommenderAgent.recommend = recommend
RecommenderAgent._update_embeddings = _update_embeddings
RecommenderAgent.update_from_session = update_from_session

print("RecommenderAgent methods implemented and applied!")
print("Your CF agent is now ready for training and evaluation.")

# ▶️ Run this to check your implementations before training
check_todo_implementations(customer_registry, item_catalogue, transaction_registry)


# 3. Training and Validation 🏋️

## 3.1 TODO 6: Choose Hyperparameters 🎛️

Set the five hyperparameters in the first cell below; the ranges to try are in the code comments. The second cell checks your values and creates the agent.

| Parameter | Meaning | Trade-off |
|---|---|---|
| `d` | Embedding dimension: length of the customer and product vectors | Complexity vs. overfitting risk |
| `lr` | Learning rate: step size of the gradient descent updates | Convergence speed vs. training stability |
| `reg` | Regularization: penalty for large embedding weights | Overfitting prevention vs. underfitting risk |
| `w_purchase` | Positive feedback weight: learning strength from purchases | How quickly the model learns from successful recommendations |
| `w_neg` | Negative feedback weight: learning strength from non-purchases | How much the model learns from rejected products |

**Hint**: Because the agent only sees feedback on 5 items per customer and iteration, it needs a fairly high learning rate (around 0.1 works well).

You judge your choice with the learning curves (3.3.1) and the test (Section 4). Tips for tuning are in 5.2.


In [ ]:
# 🎯 TODO: set your hyperparameters below and experiment with different values

# Embedding dimension (try values from 2-20)
d = None  # TODO: Choose your embedding dimension

# Learning rate (try values from 0.01-0.3) 
lr = None  # TODO: Choose your learning rate

# Regularization strength (try values from 1e-4 to 1e-1)
reg = None  # TODO: Choose your regularization parameter

# Positive feedback weight (try values from 0.1-2.0)
w_purchase = None  # TODO: Choose your positive feedback weight

# Negative feedback weight (try values from 0.1-2.0)  
w_neg = None  # TODO: Choose your negative feedback weight

In [ ]:
#@title ▶️ Run this to create your recommender agent with the hyperparameters above {display-mode: "form"}
hyperparams_set = True
hyperparams_to_check = [
    (d, "d (embedding dimension)"),
    (lr, "lr (learning rate)"),
    (reg, "reg (regularization)"),
    (w_purchase, "w_purchase (positive feedback weight)"),
    (w_neg, "w_neg (negative feedback weight)")
]

print("Checking hyperparameters...")
for param_value, param_name in hyperparams_to_check:
    if param_value is None:
        print(f"⚠️  {param_name} needs to be specified (currently None)")
        hyperparams_set = False

if not hyperparams_set:
    print("❌ Please set all hyperparameter values in the cell above before proceeding.")
    print("Replace None with your chosen values based on the guidance provided.")
else:
    cf_agent = RecommenderAgent(customer_registry, item_catalogue, transaction_registry,
                                d=d, lr=lr, reg=reg, w_purchase=w_purchase, w_neg=w_neg)

    print("✅ CF Recommender Agent created successfully!")
    print()
    print("Current CF Hyperparameters:")
    print(f"Embedding dimension (d): {cf_agent.d}")
    print(f"Learning rate (lr): {cf_agent.lr}")
    print(f"Regularization (reg): {cf_agent.reg}")
    print(f"Positive feedback weight (w_purchase): {cf_agent.w_purchase}")
    print(f"Negative feedback weight (w_neg): {cf_agent.w_neg}")
    print()
    print("Ready to proceed with training!")

## 3.2 How Training Works ⚙️

The model is trained for 300 iterations. In every iteration each customer has one session:

- **Sparse training data**: Each customer gets 15 random training products (not all 50); the other 35 are never shown to them during training (cold start). Together these are 600 of the 2,000 customer–product pairs, and every product is in the training set of about 12 customers.
- **The agent chooses what to show**: It picks K = 5 of the customer's 15 training products by expected profit $(\text{price}_i - \text{cost}_i)\cdot\hat{\kappa}(c,i)$, the score from TODO 2.
- **ε-greedy exploration**: Each of the 5 slots is filled with a random, not yet chosen training product with probability ε = 0.2.
- **Simulated purchases**: The customer buys each shown product with the true probability κ(c,i).
- **Learning**: `update_from_session()` updates the embeddings from purchases and non-purchases.

**Validation** runs every 10 iterations and at the end. It never updates the agent and rolls no dice:
- **Validation set**: all customer–product pairs outside the customer's training products (1,400 of 2,000 pairs).
- **Recommendation validation**: `agent.recommend(c, K)` on the full catalogue. We report the *expected* revenue of these recommendations (true κ · margin) as % of a perfect oracle that knows κ, compared with Highest Margin and Random on the same metric, and how many recommended products the customer never saw in training.
- **κ validation**: MSE and correlation r between κ̂ and κ on the validation pairs. Training pairs are reported separately so you can spot overfitting.

The **test** in Section 4 then runs the trained model for 100 days on the full catalogue.


### 3.2.1 Training Loop 🔁

The cell below defines and runs the training loop. You don't need to change it; read it if you want to see the details. The helpers it calls (`select_training_items`, `evaluate_validation`, ...) live in `utils/training.py`.


In [ ]:
# ▶️ Run this to define and run the training loop (no modifications needed)
TRAIN_EPSILON = 0.2  # exploration rate of the ε-greedy item choice during training
TRAIN_SEED = 42      # seed for the exploration choices (reproducible training)


def train_cf_agent_with_simulation(agent, n_iterations, customer_registry, item_catalogue,
                                   customer_ids, item_ids, true_kappa_matrix, items_per_customer,
                                   k=5, epsilon=TRAIN_EPSILON, seed=TRAIN_SEED, eval_every=10):
    """
    Train CF agent by simulating customer sessions with realistic limited item exposure.

    Each customer has a fixed set of training items (cold start: the other items are never shown
    to them during training). In every iteration the agent chooses k of these items per customer
    by expected profit (price - cost) · κ̂ with ε-greedy exploration, the customer buys each shown
    item with the true probability κ, and the agent learns from the outcome.

    Every eval_every iterations (and at the end) the agent is validated without changing it:
    agent.recommend() on the full catalogue and κ̂ vs κ on the validation pairs (items the
    customer never saw in training), see evaluate_validation().

    Args:
        agent: The CF agent to train
        n_iterations: Number of training iterations
        customer_registry: Customer registry instance
        item_catalogue: Item catalogue instance
        customer_ids: List of customer IDs
        item_ids: List of item IDs
        true_kappa_matrix: True purchase probability matrix
        items_per_customer: Number of training items per customer
        k: Number of items shown per customer and iteration
        epsilon: Probability that a shown slot is filled with a random training item
        seed: Seed for the exploration choices
        eval_every: Validation interval in iterations

    Returns:
        tuple: (history, customer_training_items)
    """
    # Create customer-specific training item subsets (REALISTIC TRAINING)
    print(f"Creating customer-specific training subsets: {items_per_customer} items per customer")
    customer_training_items = {}
    np.random.seed(42)  # For reproducible training subsets

    for customer_id in customer_ids:
        # Each customer gets their own random subset of items for training
        customer_training_items[customer_id] = np.random.choice(
            item_ids,
            size=min(items_per_customer, len(item_ids)),
            replace=False
        ).tolist()

    # Validation set = all (customer, item) pairs outside the customer's training items
    train_mask = training_mask(customer_ids, item_ids, customer_training_items)
    print(f"Training pairs: {train_mask.sum()}, validation pairs: {(~train_mask).sum()}")

    margins = item_margins(item_catalogue, item_ids)
    margin = dict(zip(item_ids, margins))
    item_index = {item_id: idx for idx, item_id in enumerate(item_ids)}
    rng = np.random.default_rng(seed)

    history = {
        "realized_revenue": [],   # revenue from the simulated purchases (dice)
        "expected_revenue": [],   # Σ true κ · margin of the shown items
        "candidate_oracle": oracle_expected_revenue(true_kappa_matrix, margins, k, train_mask),
        "references": validation_reference_revenues(item_catalogue, customer_ids, item_ids,
                                                    true_kappa_matrix, margins, k),
        "eval_iterations": [],
        "validation": [],
    }

    def validate(iteration):
        history["eval_iterations"].append(iteration)
        history["validation"].append(evaluate_validation(
            agent, customer_ids, item_ids, true_kappa_matrix, margins, train_mask, k))

    for iteration in range(n_iterations):
        if iteration % eval_every == 0:
            validate(iteration)

        realized_revenue = 0.0
        expected_revenue = 0.0
        for c_idx, customer_id in enumerate(customer_ids):
            # Agent chooses k of the customer's training items (ε-greedy)
            shown_items = select_training_items(agent, customer_id, customer_training_items[customer_id],
                                                margin, k, epsilon, rng)

            # Simulate purchases based on true κ
            purchased_items = []
            for item_id in shown_items:
                i_idx = item_index[item_id]
                true_kappa = true_kappa_matrix[c_idx, i_idx]
                expected_revenue += true_kappa * margins[i_idx]
                if np.random.random() < true_kappa:
                    purchased_items.append(item_id)
                    realized_revenue += margins[i_idx]

            # Update CF agent with feedback from the shown items
            agent.update_from_session(customer_id, shown_items, purchased_items)

        history["realized_revenue"].append(realized_revenue)
        history["expected_revenue"].append(expected_revenue)

        if iteration % 50 == 0:
            val = history["validation"][-1]
            print(f"Iteration {iteration}: training revenue ${realized_revenue:,.0f} "
                  f"(expected {100 * expected_revenue / history['candidate_oracle']:.0f}% of candidate oracle) | "
                  f"validation: recommend() {100 * val['expected_revenue'] / history['references']['Oracle']:.0f}% "
                  f"of oracle, κ r={val['val_kappa_r']:.3f}")

    validate(n_iterations)
    return history, customer_training_items


# Train CF agent with realistic limited item exposure
print("Training CF agent with realistic customer-specific item subsets...")
training_history, customer_training_items = train_cf_agent_with_simulation(
    cf_agent, N_ITERATIONS, customer_registry, item_catalogue,
    customer_ids, item_ids, true_kappa_matrix, ITEMS_PER_CUSTOMER, k=K)
print("Training simulation completed!")

## 3.3 Training Results 📊

Training is done. Now you check what your model has learned, in two steps. The learning curves in 3.3.1 show how revenue and prediction quality developed over the 300 iterations, both on the products a customer saw in training and on the products they never saw. The plots in 3.3.2 look only at the final model: how close its predicted purchase probabilities κ̂ are to the true κ for all 2,000 customer–product pairs.

Use both to judge your hyperparameters from TODO 6. Each subsection lists the values you can expect with good hyperparameters. If your results are far off, go back to 3.1, change one hyperparameter and train again.


### 3.3.1 Learning Curves 📈

The four plots show how your model develops over the 300 iterations:

- **Training Revenue per Iteration**: what the agent earns with the products it shows, compared with the best possible revenue within the training products (red dotted line).
- **Validation: Expected Revenue of Recommendations**: how good `recommend()` is on the full catalogue, in % of the global oracle that knows the true κ.
- **κ Correlation r** and **κ MSE**: how well κ̂ matches κ, on training pairs and on validation pairs the customer never saw.

With good hyperparameters the expected training revenue rises from about 58–63% to 92% of the training oracle. `recommend()` starts at the Highest Margin level (about 53% of the global oracle) and climbs to 80–85%; Random stays at about 27%. About 70% of the recommended products were never shown to that customer in training. κ̂ reaches r ≈ 0.92 (MSE ≈ 0.018) on training pairs and r ≈ 0.56–0.71 (MSE ≈ 0.06–0.075) on validation pairs. Exact numbers vary with the random seed.

**Red flags**: validation revenue stuck at the Highest Margin line, validation r near 0 while training r keeps rising (overfitting), or curves still rising steeply at iteration 300 (learning too slowly).


In [ ]:
# ▶️ Run this to plot the learning curves and track MSE over training
plot_learning_curves(training_history)

### 3.3.2 κ Prediction Quality 🔬

The three plots compare the final κ̂ with the true κ on all 2,000 customer–product pairs, training and validation pairs together:

- **Prediction Scatter**: each dot is one pair. Dots on the red diagonal are predicted exactly, dots above it are overestimated (κ̂ > κ) and dots below it underestimated.
- **Error Distribution**: how often each error κ̂ − κ occurs. The black line marks the mean error.
- **Error vs. True κ**: the same errors, plotted against the true κ.

Two numbers summarize the plots. The correlation r measures how closely κ̂ follows κ: 1 is a perfect match, 0 means no relation. The MSE is the average squared error. With good hyperparameters you should get r ≈ 0.72 and MSE ≈ 0.05. Both values are used again in the summary (5.1).


In [ ]:
# ▶️ Run this to visualize how well your model predicts purchase probabilities
correlation, mse = analyze_and_plot_kappa_prediction_quality(cf_agent, customer_ids, item_ids, true_kappa_matrix)

# 4. Test: 100 Days Against Baselines 📅

## 4.1 Baseline Strategies ⚖️

In the test your trained CF model competes against two simple strategies. Each strategy recommends K = 5 products per customer and day from the full catalogue.

| Strategy | How it chooses | Expected behaviour |
|---|---|---|
| **Random** | 5 random products | Conversion around 0.32; lowest revenue |
| **Highest Margin** | Always the 5 products with the highest margin (price − cost) | Lowest conversion (~0.27); more revenue than Random |
| **CF (Trained)** | Your model: the highest expected profit for each customer | Highest conversion (~0.65) and highest revenue |


## 4.2 Simulation and Results 🏆

The cell below runs each strategy for 100 days and plots the results; this takes 2–3 minutes. Your CF model keeps learning from the customers' purchases during the test. Unlike in training, it always shows its top 5 products and does not explore.

- **Cumulative Revenue**: total revenue so far, per strategy. The dashed line is an oracle that knows the true κ and always shows each customer their best 5 products.
- **Total Revenue and Average Conversion Rate**: final revenue next to the share of shown products that were bought.
- **Daily Conversion Rate**: the same share, day by day.
- **CF κ MSE**: how far κ̂ is from κ during the test, on all pairs and on the products shown that day.

Your model is successful if **CF > Highest Margin > Random** in total revenue and its revenue stays stable or improves. Over 100 days expect roughly CF ≈ \$1.37–1.47M, Highest Margin ≈ \$0.85M and Random ≈ \$0.44M: CF beats Highest Margin by about +60–75% and earns 85–90% of the oracle (≈ \$1.6M). The CF κ MSE on all pairs is about 0.050 on day 1 and 0.058 on day 100.


In [ ]:
# ▶️ Run this to compare your agent against the random and highest-margin baselines 
print("Running detailed agent comparison simulation...")
print("This will take 2-3 minutes to complete all simulations.")
print()

# The trained CF agent from Section 3 competes against two new baseline agents
cf_agent  # trained CF agent
cf_agent_before_test = clone_cf_agent(cf_agent)  # copy for the optional experiment in 4.3
exploration_runs = {}  # results of the optional experiment in 4.3 (ε → result)
random_agent = RandomRecommenderAgent(customer_registry, item_catalogue, transaction_registry)
highest_margin_agent = HighestMarginRecommenderAgent(customer_registry, item_catalogue, transaction_registry)

# Runs detailed simulations for comparison
simulation_results = {}

print("\nRunning detailed simulations...")
simulation_results['CF (Trained)'] = detailed_agent_simulation(
    cf_agent, 'CF (Trained)', N_SIMULATION_DAYS, K, 
    customer_registry, item_catalogue, customer_ids, item_ids, true_kappa_matrix)
simulation_results['Random'] = detailed_agent_simulation(
    random_agent, 'Random', N_SIMULATION_DAYS, K,
    customer_registry, item_catalogue, customer_ids, item_ids, true_kappa_matrix)
simulation_results['Highest Margin'] = detailed_agent_simulation(
    highest_margin_agent, 'Highest Margin', N_SIMULATION_DAYS, K,
    customer_registry, item_catalogue, customer_ids, item_ids, true_kappa_matrix)

print("\nSimulation complete! Results stored in 'simulation_results' variable.")

# Colors for consistency across plots
colors = {'CF (Trained)': '#2E86AB', 'Random': '#A23B72', 'Highest Margin': '#F18F01'}

# Create comprehensive comparison plots
oracle_daily_revenue = oracle_expected_revenue(true_kappa_matrix, item_margins(item_catalogue, item_ids), K)
plot_comprehensive_performance_analysis(simulation_results, colors, oracle_daily_revenue)

## 4.3 Optional: Exploration During the Test 🧪

In the test (4.2) CF always shows its top 5 products. In training it explored instead: with ε-greedy exploration (ε = 0.2, see 3.2) each slot sometimes showed a random product. Here you can try out what happens if CF also explores during the test.

Set `EPSILON` in the cell below and run it. The cell repeats the 100-day test from a copy of your trained model as it was before 4.2, with the same random seed for every ε. Then change `EPSILON` and run the cell again, as often as you like. Every value you try is kept, and the plot and the table compare all of them. Each new value takes about 45 seconds; values you have already run are not computed again. ε = 0 is computed once automatically as the reference. It uses a different random seed than 4.2, so its revenue differs slightly from 4.2.

This section is optional; nothing later depends on it.


In [ ]:
# ▶️ Optional: run this to re-run the test with exploration (ε-greedy, as in training)
EPSILON = 0.1  # try values between 0 and 0.5
run_exploration(EPSILON, exploration_runs, cf_agent_before_test, N_SIMULATION_DAYS, K,
                customer_registry, item_catalogue, customer_ids, item_ids, true_kappa_matrix)
plot_exploration_runs(exploration_runs, oracle_daily_revenue)
print_exploration_runs(exploration_runs)

The plot compares all ε values you have run so far, each in the same colour in all three panels:

- **Cumulative Revenue**: revenue over the 100 test days for each ε, with the oracle as in 4.2.
- **κ MSE on All Pairs**: how far κ̂ is from κ on all 2,000 pairs, day by day.
- **recommend() After the Test**: expected revenue of the model's top 5 recommendations without exploration after the 100 days, in % of the oracle.

The table lists the same results as numbers: the test revenue and its change relative to ε = 0, the κ MSE on all pairs on day 1 and day 100, the average κ MSE on the products shown, and `recommend()` after the test. Compare two things between the ε values: the revenue during the test, and the quality of the recommendations after it.


# 5. Conclusion and Next Steps 🏁

## 5.1 Summary 📋

The cell below prints a summary of your results. It first ranks the three strategies by total revenue and by conversion rate. The key findings then show how much more CF earns than each baseline, your κ correlation r and MSE from 3.3.2, the revenue and conversion of the baselines, and how the CF κ MSE and the daily revenue changed between the start and the end of the 100-day test. The business insights and the recommendation at the end are derived from these numbers.

Your model has done its job if CF ranks first by revenue and beats Highest Margin by about +60–75%, with r ≈ 0.72. If CF does not beat both baselines, go back to TODO 6 (3.1) and use the tips in 5.2. The summary gives you the numbers; explaining them, with the help of the plots in 3.3 and 4.2, is up to you.


In [ ]:
# ▶️ Run this to generate the executive summary and business recommendations
if 'correlation' not in dir() or 'mse' not in dir():
    correlation, mse = analyze_and_plot_kappa_prediction_quality(cf_agent, customer_ids, item_ids, true_kappa_matrix)
print_comprehensive_analysis(simulation_results, correlation, mse, customer_registry)

## 5.2 Improve Your Model 🚀

**If you haven't passed the baselines yet**: Go back to TODO 6 (3.1), adjust the hyperparameters (learning rate, embedding dimension, regularization, feedback weights) and re-run Sections 3 to 5.

**If you've already passed the baselines**, you can try to improve further:
- Fine-tune your hyperparameters
- Implement early stopping or learning rate scheduling
- Analyze error patterns (3.3.2) to identify areas for improvement
- Test different optimization algorithms (Adam, SGD, RMSprop, etc.)
- Optional, advanced: train multiple CF models with different hyperparameters and average their predictions (ensemble). This is by no means needed for the assignment but can improve robustness.

**Tips for experimentation**:
- Start with values that work and change **one hyperparameter at a time** to isolate its effect.
- Use the training curves and validation metrics (3.3.1) to guide your choices: training curves should show smooth, steady improvement.
- Keep notes on what you tried and the results.
- Re-run your baseline settings to confirm reproducibility.

Good luck improving your model! 🎉
